In [10]:
%load_ext autoreload
%autoreload 2

import numpy as np
import xarray as xr
import geoviews as gv
import geoviews.feature as gf
import cartopy.crs as ccrs
import cartopy.io.shapereader as shapereader
import matplotlib.pyplot as plt
import shapely.geometry
import scipy.constants
import geopandas as gpd
import geoarrow
from tqdm import tqdm

import requests
import time

import xopr

import holoviews as hv
import hvplot.xarray
import hvplot.pandas
hvplot.extension('bokeh')

# import util functions
import sys
sys.path.append("..")
from xover.echopower_util import crossover_echo_power

# data dir
data_dir = "../data/"
df_xover = gpd.read_parquet(f"{data_dir}icesat-radar-merged_points.parquet")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
# Useful background features for maps
background_map = gf.ocean.opts(projection=ccrs.SouthPolarStereo(), scale='50m') * gf.coastline.opts(projection=ccrs.SouthPolarStereo(), scale='50m')
opr = xopr.OPRConnection(cache_dir="radar_cache")

regions_df = xopr.geometry.get_antarctic_regions(merge_regions=False).to_crs('EPSG:3031')
regions_df.hvplot(frame_width=600, aspect='equal', hover_cols=['NAME'], c='TYPE')

region = xopr.geometry.get_antarctic_regions(name="Thwaites", merge_regions=True, simplify_tolerance=100)
region_projected = xopr.geometry.project_geojson(region, source_crs='EPSG:4326', target_crs="EPSG:3031")

region_hv = hv.Polygons([region_projected]).opts(
    color='green',
    line_color='black',
    fill_alpha=0.5)

(background_map * region_hv).opts(aspect='equal')

stac_items_df = opr.query_frames(geometry=region, date_range="2005-01-01T00:00:00Z/2025-06-01T00:00:00Z") # Can also add a date range: date_range="2020-01-01T00:00:00Z/2024-01-01T00:00:00Z"
stac_items_df = stac_items_df.to_crs('EPSG:3031')

print(f"Found {len(stac_items_df)} frames across {stac_items_df['collection'].nunique()} collections:")
stac_items_df.groupby('collection').size()

flight_lines = stac_items_df.hvplot(by='collection')
(background_map * region_hv * flight_lines).opts(frame_width=500, aspect='equal', active_tools=['pan', 'wheel_zoom'])

Found 828 frames across 9 collections:


:Overlay
   .Ocean.I     :Feature   [Longitude,Latitude]
   .Coastline.I :Feature   [Longitude,Latitude]
   .Polygons.I  :Polygons   [x,y]
   .NdOverlay.I :NdOverlay   [collection]
      :Path   [x,y]

In [14]:
intersections_extracted = crossover_echo_power(df_xover, stac_items_df)

Processing 106 intersections


AttributeError: 'Series' object has no attribute 'intersection_geometry'